# The Same Agent, Raw vs PydanticAI

**What you will build:** the exact weather-and-calculator agent from notebook 0.3 — but in PydanticAI. Side by side, you'll see precisely what the framework does for you, so you adopt it knowing what it hides.

This is a *Rosetta Stone* notebook: same task, two languages.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/10_rosetta_raw_vs_pydanticai.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

```{note}
OpenRouter speaks the OpenAI Chat Completions format, so we use `OpenAIProvider` with OpenRouter's `base_url` — the same pattern you used with the raw SDK in Block 0. One consistent way to reach any model, across all three blocks.
```

## Reminder: what we wrote by hand in 0.3

To make that agent, we wrote all of this ourselves:

- a `while` loop with a turn limit,
- a JSON schema describing every tool,
- code to read `tool_calls`, parse the JSON `arguments`, dispatch to the right function,
- code to append the assistant message and each `tool` result back into `messages`.

That was the point — you should know it cold. Now watch it collapse.

## The same agent in PydanticAI

Tools are just Python functions with a docstring; you register them with `@agent.tool_plain`. The loop, the schemas, and the message plumbing are gone.

In [ ]:
agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@agent.tool_plain
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    fake = {"bilbao": "18C, light rain", "madrid": "31C, sunny", "london": "14C, cloudy"}
    return fake.get(city.lower(), "20C, clear")

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def safe_eval(expr):                       # same helper as notebook 0.3 -- no eval(), so 9**9**9 can't hang the kernel
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError('unsupported')
    return ev(ast.parse(expr, mode='eval'))

@agent.tool_plain
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * (3 + 4)'."""
    try:
        return str(safe_eval(expression))
    except Exception:
        return "Error: only +, -, *, / and parentheses are supported."

result = await agent.run("What's the weather in Madrid, and what is 231 * 17?")
print(result.output)

Same answer as 0.3, a third of the code. Notice what you did **not** write:

| In 0.3 you wrote by hand | In PydanticAI |
|--------------------------|---------------|
| The `while` loop + turn limit | Handled by `agent.run()` |
| A JSON schema per tool | Generated from the type hints |
| Parsing `tool_calls` and JSON arguments | Automatic |
| Appending tool results to `messages` | Automatic |
| `str` in, `str` out | Typed results (next notebooks) |

```{note}
The framework didn't add magic — it removed boilerplate. Everything it does, you already did by hand in Block 0. That's why we built the raw version first: you can now debug PydanticAI, because you know what it's doing underneath.
```

::::{dropdown} 🔍 What PydanticAI kept from the raw version
:color: info

It's the *same* protocol from 0.0, just automated:

```
you:   @agent.tool_plain def calculator(...)      -> PydanticAI builds the JSON schema
run:   await agent.run(question)                  -> PydanticAI runs the THINK/ACT/OBSERVE loop
          model asks for calculator(...)          ->   PydanticAI parses args, calls your function
          model asks for get_weather(...)         ->   ... appends results, loops
          model returns text                      ->   loop ends
result.output                                     -> the final answer
```
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a third tool.** Write `word_count(text: str) -> int` in both versions — the raw loop from 0.3 and this PydanticAI one — and compare how much boilerplate each addition costs.
2. **Swap the model.** Change `MODEL_NAME` to a Claude or GPT id and confirm the same code runs unchanged.
::::

**What's next:** in **1.1** we slow down and cover the `Agent` object properly — system prompts, running, and the result object.